Directory three of the project:

```
├── code
├── data
├── jmxs
├── model
├── pcaps
└── results
```

The working dir of the script is the code/ directory

The code will be used to extract data to instrument the model. The data has been obtained from the system executing the application. The perform N readings in a observation time interval T. 

    1. read data from the data dir/ - 
    
    ```
    ├── 10-38
    │   ├── containers_pre.json
    │   ├── containers_post.json
    │   ├── energy.csv
    │   ├── system_pre.json
    │   ├── system_post.json
    │   └── requests.jtl
    ├── 11-38
    │   ├── containers_pre.json
    │   ├── containers_post.json
    │   ├── energy.csv
    │   ├── system_pre.json
    │   ├── system_post.json
    │   └── requests.jtl
    ```
       
    The name of the directory contains two numbers. The former describes the ith reading while the second one the population of customers operating while reading performance and energy values.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from itertools import chain
import csv, json, glob, os

In [2]:
DATA = f"../data/one-shot-2"
DIRS = list(map(lambda x: f"{DATA}/{x}", os.listdir(DATA)))

In [3]:
DIRS75 = list(filter(lambda x: x.find("-75") != -1, DIRS))

In [4]:
def get_energy_stats(trials):
    time   = np.array([len(x['time']) for x in trials])
    energy = np.array(
        [np.trapz(x['power'], x['time']) for x in trials]
    )
    e = energy/time
    
    print(
            f"# Energy Per Visit(Joule/Visit):\n",
            f"## Mean:\t\t\t{energy.mean()}", 
            f"## Min-Max:\t\t\t[{energy.min()}, {energy.max()}]",
            f"## Var:\t\t\t\t{energy.var()}", 
            f"## Std:\t\t\t\t{energy.std()}", 
            '\n'
            f"# Average Response Time:\t{time.mean()}",
            f"# e (Joule/s):\t\t\t{e.mean()}, [{e.min()}, {e.max()}]",
            sep='\n'
         )

In [5]:
ENERGYFILES = [f"{x}/energy.csv" for x in DIRS]

In [6]:
energy_values = [
    pd.read_csv(x, names = ["time", "power"]) for x in ENERGYFILES
]

In [7]:
get_energy_stats(energy_values)

# Energy Per Visit(Joule/Visit):

## Mean:			844.0964277777778
## Min-Max:			[532.05755, 1385.1209999999999]
## Var:				87873.73628263561
## Std:				296.4350456383921

# Average Response Time:	14.733333333333333
# e (Joule/s):			55.85692211178587, [47.4608375, 66.76880789473684]


In [8]:
from helpers.stats import SystemStats
from helpers.stats import ContainerStats

In [9]:
def calculate_containers_utilization(DIRS):
    rows = []
    for x in DIRS:
        f1, f2 = glob.glob(f"{x}/containers_*.json")
        rows.append(
            {k:v['cpu'] for k,v in ContainerStats(f2, f1).data.items()}
        )

    return pd.DataFrame.from_records(rows)

def calculate_system_utilization(DIRS):
    rows = []
    for x in DIRS:
        f1, f2 = glob.glob(f"{x}/system_*.json")
        rows.append({'cpu': SystemStats(f2, f1).data[0]['cpu0']})

    return pd.DataFrame.from_records(rows)

In [16]:
containers_util = calculate_containers_utilization(DIRS75)

In [31]:
containers_util.head(1).values.sum()/8

33.7388707523015

In [22]:
np.array(containers_util['baseline-ts-ui-dashboard-1']).mean()

1.7342660447536293

In [23]:
np.array(containers_util['baseline-ts-travel-service-1']).mean()

26.377119123105526

In [24]:
np.array(calculate_system_utilization(DIRS75)).mean()

46.043166666666664